# 面试题：不用图框架，怎样实现 GNN 消息传递与节点分类？

## 可以直接复述的回答

GNN 的核心是把邻居特征按图结构聚合，再与节点自身特征组合并学习分类边界。实现时应明确是否加自环、邻接矩阵怎样归一化、消息方向和每层张量形状，否则模型含义会悄悄改变。只看节点自身的规则无法识别“自身指标正常但处于高风险团伙”的节点，也会误伤“指标偏高但邻居正常”的节点。一个最小 GraphSAGE 层可以手写为 own×W_self + neighbor_mean×W_neighbor，随后 ReLU 和线性输出。训练要只在训练 mask 上计算 loss，但消息可以经过无标签邻居，并实际执行 forward、backward 与参数更新。层数过深还会过度平滑，使不同节点表示趋同；残差或限制层数只能缓解而非彻底解决。本题用 10 个商户节点和交易关联边完成真实梯度训练、逐节点概率与过平滑实验。

## 真实案例

节点特征为脱敏的拒付率、异常设备比例，标签为正常或高风险；图边表示共享结算设备。训练标签仅覆盖 6 个节点，测试 4 个刻意具有“自身特征与邻居信号冲突”的节点。数据是机制教学样本，不能用于真实风控决策。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示节点和训练结果
import torch  # 导入 PyTorch 以执行真实图张量前向与反向传播
torch.manual_seed(11)  # 固定参数初始化保证实验可复现
torch.set_num_threads(1)  # 限制 CPU 线程以稳定小实验执行
nodes = [{"id": "F1", "拒付率": 0.90, "异常设备": 0.80, "label": 1, "split": "train"}, {"id": "F2", "拒付率": 0.85, "异常设备": 0.75, "label": 1, "split": "train"}, {"id": "F3", "拒付率": 0.80, "异常设备": 0.90, "label": 1, "split": "train"}, {"id": "F4", "拒付率": 0.20, "异常设备": 0.25, "label": 1, "split": "test"}, {"id": "F5", "拒付率": 0.30, "异常设备": 0.20, "label": 1, "split": "test"}, {"id": "N1", "拒付率": 0.10, "异常设备": 0.15, "label": 0, "split": "train"}, {"id": "N2", "拒付率": 0.15, "异常设备": 0.10, "label": 0, "split": "train"}, {"id": "N3", "拒付率": 0.20, "异常设备": 0.15, "label": 0, "split": "train"}, {"id": "N4", "拒付率": 0.75, "异常设备": 0.70, "label": 0, "split": "test"}, {"id": "N5", "拒付率": 0.65, "异常设备": 0.75, "label": 0, "split": "test"}]  # 构造十个含训练测试标记的商户节点
edges = [("F1", "F2"), ("F1", "F3"), ("F2", "F3"), ("F2", "F4"), ("F3", "F4"), ("F3", "F5"), ("F4", "F5"), ("N1", "N2"), ("N1", "N3"), ("N2", "N3"), ("N2", "N4"), ("N3", "N4"), ("N3", "N5"), ("N4", "N5"), ("F1", "N1")]  # 构造团伙内密集连接和一条跨群噪声边
node_to_index = {node["id"]: index for index, node in enumerate(nodes)}  # 建立节点 ID 到矩阵行号的映射
features = torch.tensor([[node["拒付率"], node["异常设备"]] for node in nodes], dtype=torch.float32)  # 构造十乘二节点特征矩阵
targets = torch.tensor([node["label"] for node in nodes], dtype=torch.long)  # 构造全部节点人工标签张量
train_mask = torch.tensor([node["split"] == "train" for node in nodes], dtype=torch.bool)  # 构造只监督六个训练节点的掩码
test_mask = ~train_mask  # 构造四个测试节点掩码
print("节点输入预览：")  # 输出真实图节点标题
pprint(nodes)  # 展示特征、标签和切分
print("共享设备边：", edges)  # 展示图结构输入

节点输入预览：
[{'id': 'F1', 'label': 1, 'split': 'train', '异常设备': 0.8, '拒付率': 0.9},
 {'id': 'F2', 'label': 1, 'split': 'train', '异常设备': 0.75, '拒付率': 0.85},
 {'id': 'F3', 'label': 1, 'split': 'train', '异常设备': 0.9, '拒付率': 0.8},
 {'id': 'F4', 'label': 1, 'split': 'test', '异常设备': 0.25, '拒付率': 0.2},
 {'id': 'F5', 'label': 1, 'split': 'test', '异常设备': 0.2, '拒付率': 0.3},
 {'id': 'N1', 'label': 0, 'split': 'train', '异常设备': 0.15, '拒付率': 0.1},
 {'id': 'N2', 'label': 0, 'split': 'train', '异常设备': 0.1, '拒付率': 0.15},
 {'id': 'N3', 'label': 0, 'split': 'train', '异常设备': 0.15, '拒付率': 0.2},
 {'id': 'N4', 'label': 0, 'split': 'test', '异常设备': 0.7, '拒付率': 0.75},
 {'id': 'N5', 'label': 0, 'split': 'test', '异常设备': 0.75, '拒付率': 0.65}]
共享设备边： [('F1', 'F2'), ('F1', 'F3'), ('F2', 'F3'), ('F2', 'F4'), ('F3', 'F4'), ('F3', 'F5'), ('F4', 'F5'), ('N1', 'N2'), ('N1', 'N3'), ('N2', 'N3'), ('N2', 'N4'), ('N3', 'N4'), ('N3', 'N5'), ('N4', 'N5'), ('F1', 'N1')]


## Baseline / 基线：只看节点自身拒付率

规则把拒付率大于 0.5 的节点判为高风险。它会漏掉被团伙包围的 F4/F5，也会误伤邻居正常的 N4/N5，正好体现关系信号的价值。

In [2]:
baseline_predictions = (features[:, 0] > 0.5).to(torch.long)  # 用拒付率阈值生成节点自身基线预测
baseline_test_accuracy = float((baseline_predictions[test_mask] == targets[test_mask]).to(torch.float32).mean())  # 计算四个测试节点准确率
baseline_rows = []  # 创建逐测试节点基线结果表
for index, node in enumerate(nodes):  # 遍历全部商户节点
    if node["split"] == "test":  # 只记录未参与监督的测试节点
        baseline_rows.append({"节点": node["id"], "拒付率": node["拒付率"], "真实": node["label"], "Baseline预测": int(baseline_predictions[index])})  # 保存规则证据和预测
print("仅节点特征 Baseline：")  # 输出基线标题
pprint(baseline_rows)  # 展示四个冲突节点的错误决策
print(f"Baseline test accuracy={baseline_test_accuracy:.3f}")  # 输出同测试集基线指标

仅节点特征 Baseline：
[{'Baseline预测': 0, '拒付率': 0.2, '真实': 1, '节点': 'F4'},
 {'Baseline预测': 0, '拒付率': 0.3, '真实': 1, '节点': 'F5'},
 {'Baseline预测': 1, '拒付率': 0.75, '真实': 0, '节点': 'N4'},
 {'Baseline预测': 1, '拒付率': 0.65, '真实': 0, '节点': 'N5'}]
Baseline test accuracy=0.000


## 手写邻接矩阵与邻居均值

邻接矩阵不含自环，因为模型会用独立 W_self 处理自身特征。每行除以度数得到邻居均值；孤立节点用零消息，生产实现则需显式定义其退化策略。

In [3]:
adjacency = torch.zeros((len(nodes), len(nodes)), dtype=torch.float32)  # 创建十乘十无向邻接矩阵
for left, right in edges:  # 遍历共享设备边
    left_index = node_to_index[left]  # 查找左侧节点行号
    right_index = node_to_index[right]  # 查找右侧节点行号
    adjacency[left_index, right_index] = 1.0  # 写入正向连接
    adjacency[right_index, left_index] = 1.0  # 写入反向连接
degrees = adjacency.sum(dim=1, keepdim=True)  # 计算每个节点的邻居数量
normalized_adjacency = adjacency / degrees.clamp_min(1.0)  # 按行归一化得到邻居均值算子
neighbor_features = normalized_adjacency @ features  # 手算每个节点收到的均值消息
message_rows = []  # 创建自身特征与邻居消息对照表
for index, node in enumerate(nodes):  # 遍历全部图节点
    message_rows.append({"节点": node["id"], "自身": [round(float(value), 3) for value in features[index]], "邻居均值": [round(float(value), 3) for value in neighbor_features[index]], "度": int(degrees[index])})  # 保存可解释的聚合中间量
print("每个节点的自身特征与邻居均值消息：")  # 输出消息传递标题
pprint(message_rows)  # 展示冲突节点如何从邻居获得相反证据

每个节点的自身特征与邻居均值消息：
[{'度': 3, '自身': [0.9, 0.8], '节点': 'F1', '邻居均值': [0.583, 0.6]},
 {'度': 3, '自身': [0.85, 0.75], '节点': 'F2', '邻居均值': [0.633, 0.65]},
 {'度': 4, '自身': [0.8, 0.9], '节点': 'F3', '邻居均值': [0.562, 0.5]},
 {'度': 3, '自身': [0.2, 0.25], '节点': 'F4', '邻居均值': [0.65, 0.617]},
 {'度': 2, '自身': [0.3, 0.2], '节点': 'F5', '邻居均值': [0.5, 0.575]},
 {'度': 3, '自身': [0.1, 0.15], '节点': 'N1', '邻居均值': [0.417, 0.35]},
 {'度': 3, '自身': [0.15, 0.1], '节点': 'N2', '邻居均值': [0.35, 0.333]},
 {'度': 4, '自身': [0.2, 0.15], '节点': 'N3', '邻居均值': [0.412, 0.425]},
 {'度': 3, '自身': [0.75, 0.7], '节点': 'N4', '邻居均值': [0.333, 0.333]},
 {'度': 2, '自身': [0.65, 0.75], '节点': 'N5', '邻居均值': [0.475, 0.425]}]


## 手写 GraphSAGE 前向、交叉熵与真实反向传播

模型没有调用图神经网络包：邻居聚合就是上面的矩阵乘法，自身和邻居分别映射到隐藏层。loss 只读取 train_mask，测试标签完全不参与参数更新。

In [4]:
class MeanSAGE(torch.nn.Module):  # 定义一层均值聚合图分类器
    def __init__(self, input_size, hidden_size, class_count):  # 初始化自身、邻居和输出参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.self_weight = torch.nn.Parameter(torch.randn(input_size, hidden_size) * 0.2)  # 创建自身特征映射权重
        self.neighbor_weight = torch.nn.Parameter(torch.randn(input_size, hidden_size) * 0.2)  # 创建邻居消息映射权重
        self.hidden_bias = torch.nn.Parameter(torch.zeros(hidden_size))  # 创建隐藏层偏置
        self.output_weight = torch.nn.Parameter(torch.randn(hidden_size, class_count) * 0.2)  # 创建隐藏表示到类别的权重
        self.output_bias = torch.nn.Parameter(torch.zeros(class_count))  # 创建输出类别偏置
    def forward(self, node_features, mean_adjacency):  # 定义消息聚合和节点分类前向过程
        neighbor_messages = mean_adjacency @ node_features  # 从邻接矩阵计算邻居均值消息
        hidden_pre_activation = 0.05 * (node_features @ self.self_weight) + neighbor_messages @ self.neighbor_weight + self.hidden_bias  # 在团伙场景中降低易伪装自身特征并突出邻居消息
        hidden = torch.relu(hidden_pre_activation)  # 用 ReLU 得到节点隐藏表示
        logits = hidden @ self.output_weight + self.output_bias  # 把隐藏表示映射为两类 logits
        return logits, neighbor_messages, hidden  # 返回分类分数与可解释中间张量
def cross_entropy(logits, labels):  # 定义数值稳定的手写交叉熵
    log_probabilities = logits - torch.logsumexp(logits, dim=1, keepdim=True)  # 计算归一化对数概率
    selected = log_probabilities[torch.arange(labels.shape[0]), labels]  # 选择每个训练节点真实类别的对数概率
    return -selected.mean()  # 返回训练节点平均负对数似然
model = MeanSAGE(2, 6, 2)  # 实例化最小图神经网络
training_ledger = []  # 创建 loss 与梯度账本
for epoch in range(401):  # 执行四百零一次全图前向训练
    logits, messages, hidden = model(features, normalized_adjacency)  # 运行真实 GNN forward
    loss = cross_entropy(logits[train_mask], targets[train_mask])  # 只在六个训练节点上计算监督损失
    loss.backward()  # 运行真实 backward 计算全部参数梯度
    gradient_norm = float(model.neighbor_weight.grad.norm())  # 读取邻居权重梯度证明图消息参与训练
    if epoch % 80 == 0:  # 每八十轮记录一次训练状态
        training_ledger.append({"epoch": epoch, "loss": round(float(loss), 5), "neighbor_grad_norm": round(gradient_norm, 5)})  # 保存损失和邻居梯度范数
    with torch.no_grad():  # 关闭参数更新阶段的梯度追踪
        for parameter in model.parameters():  # 遍历全部手写参数张量
            parameter -= 0.08 * parameter.grad  # 手动执行 SGD 参数更新
    for parameter in model.parameters():  # 遍历全部参数准备下一轮
        parameter.grad.zero_()  # 清零梯度避免跨轮累加
print("GNN 训练 loss 与邻居权重梯度：")  # 输出真实训练过程标题
pprint(training_ledger)  # 展示损失下降和梯度变化

GNN 训练 loss 与邻居权重梯度：
[{'epoch': 0, 'loss': 0.69535, 'neighbor_grad_norm': 0.0179},
 {'epoch': 80, 'loss': 0.68966, 'neighbor_grad_norm': 0.02757},
 {'epoch': 160, 'loss': 0.67954, 'neighbor_grad_norm': 0.03153},
 {'epoch': 240, 'loss': 0.65135, 'neighbor_grad_norm': 0.05035},
 {'epoch': 320, 'loss': 0.58713, 'neighbor_grad_norm': 0.06676},
 {'epoch': 400, 'loss': 0.47348, 'neighbor_grad_norm': 0.07895}]


## 逐节点结果与结果解读

F4/F5 自身指标低，但邻居均值高；N4/N5 正好相反。模型在训练节点上学会分别使用自身和邻居证据后，可以纠正只看阈值的错误。概率只是教学模型置信，不是可直接使用的风控概率。

In [5]:
def softmax(logits):  # 定义不调用封装分类器的概率转换
    shifted = logits - logits.max(dim=1, keepdim=True).values  # 减去每行最大值避免指数溢出
    exponentials = shifted.exp()  # 对稳定后的 logits 取指数
    return exponentials / exponentials.sum(dim=1, keepdim=True)  # 按节点归一化为两类概率
with torch.no_grad():  # 关闭评估阶段梯度记录
    final_logits, final_messages, final_hidden = model(features, normalized_adjacency)  # 运行训练完成后的 GNN 前向
    probabilities = softmax(final_logits)  # 把 logits 转换为风险概率
    predictions = probabilities.argmax(dim=1)  # 读取每个节点概率最高类别
gnn_test_accuracy = float((predictions[test_mask] == targets[test_mask]).to(torch.float32).mean())  # 计算四个测试节点准确率
result_rows = []  # 创建逐测试节点模型结果表
for index, node in enumerate(nodes):  # 遍历全部节点
    if node["split"] == "test":  # 仅展示测试节点避免混淆训练拟合
        result_rows.append({"节点": node["id"], "真实": node["label"], "Baseline": int(baseline_predictions[index]), "GNN": int(predictions[index]), "风险概率": round(float(probabilities[index, 1]), 4), "邻居拒付均值": round(float(final_messages[index, 0]), 3)})  # 保存预测、概率与邻居证据
print("测试节点逐样本结果：")  # 输出结果表标题
pprint(result_rows)  # 展示 GNN 如何纠正四个冲突样本
print(f"test accuracy 从 {baseline_test_accuracy:.3f} 提升到 {gnn_test_accuracy:.3f}")  # 输出同测试集指标对照

测试节点逐样本结果：
[{'Baseline': 0, 'GNN': 1, '真实': 1, '节点': 'F4', '邻居拒付均值': 0.65, '风险概率': 0.6973},
 {'Baseline': 0, 'GNN': 1, '真实': 1, '节点': 'F5', '邻居拒付均值': 0.5, '风险概率': 0.596},
 {'Baseline': 1,
  'GNN': 0,
  '真实': 0,
  '节点': 'N4',
  '邻居拒付均值': 0.333,
  '风险概率': 0.3499},
 {'Baseline': 1, 'GNN': 0, '真实': 0, '节点': 'N5', '邻居拒付均值': 0.475, '风险概率': 0.486}]
test accuracy 从 0.000 提升到 1.000


## 失败案例：重复均值传播导致过度平滑

如果不加参数和非线性，只把邻居均值反复传播很多层，连通图中的节点表示会逐渐相似，风险团伙和正常群体的距离缩小。下面用特征方差量化塌缩，再用残差保留一部分原始状态；残差只缓解问题，生产上还需层数、跳连和采样设计。

In [6]:
def repeated_smoothing(initial_features, operator, layers, residual_weight=0.0):  # 定义纯消息传播的过平滑实验
    state = initial_features.clone()  # 复制初始节点特征
    for _ in range(layers):  # 重复执行指定层数的均值传播
        neighbor_state = operator @ state  # 计算当前层邻居均值
        state = residual_weight * state + (1 - residual_weight) * neighbor_state  # 用残差比例组合旧状态和邻居消息
    return state  # 返回多层传播后的节点表示
plain_smoothed = repeated_smoothing(features, normalized_adjacency, 20, residual_weight=0.0)  # 运行无残差的二十层传播
residual_smoothed = repeated_smoothing(features, normalized_adjacency, 20, residual_weight=0.8)  # 运行保留八成自身状态的传播
initial_variance = float(features.var(dim=0).mean())  # 计算原始节点间平均特征方差
plain_variance = float(plain_smoothed.var(dim=0).mean())  # 计算过度平滑后的节点方差
residual_variance = float(residual_smoothed.var(dim=0).mean())  # 计算残差修正后的节点方差
print("失败案例：二十层均值传播的表示方差", {"初始": round(initial_variance, 6), "无残差": round(plain_variance, 6), "有残差": round(residual_variance, 6)})  # 展示表示塌缩及缓解幅度
print("F4 与 N4 的无残差表示：", plain_smoothed[node_to_index["F4"]].tolist(), plain_smoothed[node_to_index["N4"]].tolist())  # 展示冲突节点表示趋同

失败案例：二十层均值传播的表示方差 {'初始': 0.106958, '无残差': 0.000599, '有残差': 0.006937}
F4 与 N4 的无残差表示： [0.5178054571151733, 0.5053834319114685] [0.46552830934524536, 0.4512844681739807]


## 生产差距

真实图需要时间切分、边类型与方向、采样邻居、孤立节点策略和动态图更新。标签传播可能产生同群泄漏，风控场景还要防止未来边进入训练、对抗团伙改变拓扑，并做概率校准、分群召回、解释审计与人工复核；大图训练通常需要稀疏算子和分布式采样。

In [7]:
assert len(nodes) == 10 and len(edges) == 15  # 验证案例包含十个节点和十五条真实语义边
assert training_ledger[-1]["loss"] < training_ledger[0]["loss"]  # 验证真实反向传播使监督损失下降
assert gnn_test_accuracy > baseline_test_accuracy  # 验证图消息改善冲突节点分类
assert gnn_test_accuracy == 1.0  # 验证四个教学测试节点全部分类正确
assert plain_variance < initial_variance  # 验证深层均值传播确实压缩节点差异
assert residual_variance > plain_variance  # 验证残差连接保留更多节点差异
print("最小回归测试通过：消息聚合、真实梯度、节点分类与过平滑修正均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：消息聚合、真实梯度、节点分类与过平滑修正均满足预期
